# Car Model Recognition — Kaggle Training
BIL462 | Group: Junkers

**Gereksinimler:**
- Accelerator: GPU T4 x2
- Internet: On
- Dataset: `rickyyyyyyy/torchvision-stanford-cars` eklenmiş olmalı

In [ ]:
import subprocess, sys

subprocess.run(['pip', 'install', '-q', 'timm'], check=True)
subprocess.run([
    'git', 'clone', '-q',
    'https://github.com/umutgokmen/ann-project',
    '/kaggle/working/ann-project'
], check=True)
sys.path.insert(0, '/kaggle/working/ann-project')
print('Setup complete')

In [ ]:
import os, yaml
from pathlib import Path

DATASET_INPUT = Path('/kaggle/input/datasets/rickyyyyyyy/torchvision-stanford-cars/stanford_cars')
WORK_DIR      = Path('/kaggle/working/ann-project')

print('Dataset contents:', [p.name for p in DATASET_INPUT.iterdir()])

In [ ]:
with open(WORK_DIR / 'configs' / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['dataset_dir']       = str(DATASET_INPUT)
cfg['data']['num_workers']       = 2
cfg['logging']['checkpoint_dir'] = '/kaggle/working/checkpoints'
cfg['logging']['log_dir']        = '/kaggle/working/logs'
cfg['training']['batch_size']    = 64
cfg['training']['epochs']        = 40
cfg['training']['amp']           = True

KAGGLE_CONFIG = '/kaggle/working/config_kaggle.yaml'
with open(KAGGLE_CONFIG, 'w') as f:
    yaml.dump(cfg, f)

print(f"dataset_dir : {cfg['data']['dataset_dir']}")
print(f"batch_size  : {cfg['training']['batch_size']}")
print(f"epochs      : {cfg['training']['epochs']}")
print(f"backbone    : {cfg['model']['backbone']}")

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
os.chdir(WORK_DIR)
from src.dataset import build_dataloaders
from src.model import build_model

loaders = build_dataloaders(cfg)
print(f"Train : {len(loaders['train'])} batch")
print(f"Val   : {len(loaders['val'])} batch")
print(f"Test  : {len(loaders['test'])} batch")
print(f"Class : {len(loaders['class_names'])}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = build_model(cfg).to(device)

images, labels = next(iter(loaders['train']))
with torch.no_grad():
    out = model(images.to(device))
print(f'Forward OK | input: {images.shape} | output: {out.shape}')

In [ ]:
import subprocess
result = subprocess.run(
    ['python', 'src/train.py', '--config', KAGGLE_CONFIG],
    cwd=str(WORK_DIR)
)
print('Training exit code:', result.returncode)

In [ ]:
result = subprocess.run(
    [
        'python', 'src/evaluate.py',
        '--config', KAGGLE_CONFIG,
        '--checkpoint', '/kaggle/working/checkpoints/best.pt',
        '--output_dir', '/kaggle/working/evaluation',
    ],
    cwd=str(WORK_DIR)
)
print('Evaluation exit code:', result.returncode)

In [ ]:
from IPython.display import Image as IPImage
IPImage('/kaggle/working/evaluation/confusion_matrix.png')

In [ ]:
# List output files (checkpoints + evaluation)
import os
for root, dirs, files in os.walk('/kaggle/working'):
    dirs[:] = [d for d in dirs if d not in ['ann-project', '__pycache__']]
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f'{path.replace("/kaggle/working/", "")}  ({size // 1024}KB)')